In [2]:
from pyspark.sql import SparkSession

In [4]:
Spark = (
    SparkSession.builder.appName("TP_RDD").master("local[*]").getOrCreate()
)

sc = Spark.sparkContext
print("Spark Session crée avec Succes")

Spark Session crée avec Succes


In [6]:
print(sc.version)

4.0.0


In [17]:
DATA = os.path.abspath("data")

os.makedirs(DATA, exist_ok=True)

print("Dossier des données :", DATA)
print("Dossier accessible :", os.path.exists(DATA))

Dossier des données : /home/jovyan/data
Dossier accessible : True


In [18]:
lignes = """id,date,ville,produit,quantite,prix
1,2025-01-05,Casablanca,Laptop,2,9500
2,2025-01-06,Rabat,Telephone,5,3200
3,2025-01-07,Casablanca,Telephone,3,3200
4,2025-01-09,Fes,Tablette,4,2800
5,2025-01-11,Rabat,Laptop,1,9500
6,2025-01-12,Marrakech,Tablette,6,2800
7,2025-01-15,Casablanca,Souris,20,120
8,2025-01-16,Fes,Laptop,2,9500
9,2025-01-18,Marrakech,Telephone,4,3200
10,2025-01-20,Rabat,Souris,15,120
11,2025-01-21,Agadir,Clavier,10,250
12,ligne_corrompue
13,2025-01-22,Casablanca,Tablette,2,abc
14,2025-01-25,Fes,Souris,30,120
15,2025-01-26,Agadir,Clavier,8,250
16,2025-01-27,Casablanca,Clavier,12,250
17,2025-02-02,Rabat,Telephone,2,3200
18,2025-02-03,Agadir,Laptop,1,9500
19,2025-02-04,Marrakech,Souris,,120
20,2025-02-05,Rabat,Tablette,3,2800
21,2025-02-07,Marrakech,Laptop,2,9500
22,2025-02-09,Fes,Clavier,5,250"""

with open(os.path.join(DATA, "ventes.csv"), "w") as f:
    f.write(lignes + "\n")

print("Fichier créé :", os.path.exists(os.path.join(DATA, "ventes.csv")))

Fichier créé : True


In [19]:
print(os.listdir(DATA))

with open(os.path.join(DATA, "ventes.csv"), "r") as f:
    print(f.read())

['.ipynb_checkpoints', 'ventes.csv']
id,date,ville,produit,quantite,prix
1,2025-01-05,Casablanca,Laptop,2,9500
2,2025-01-06,Rabat,Telephone,5,3200
3,2025-01-07,Casablanca,Telephone,3,3200
4,2025-01-09,Fes,Tablette,4,2800
5,2025-01-11,Rabat,Laptop,1,9500
6,2025-01-12,Marrakech,Tablette,6,2800
7,2025-01-15,Casablanca,Souris,20,120
8,2025-01-16,Fes,Laptop,2,9500
9,2025-01-18,Marrakech,Telephone,4,3200
10,2025-01-20,Rabat,Souris,15,120
11,2025-01-21,Agadir,Clavier,10,250
12,ligne_corrompue
13,2025-01-22,Casablanca,Tablette,2,abc
14,2025-01-25,Fes,Souris,30,120
15,2025-01-26,Agadir,Clavier,8,250
16,2025-01-27,Casablanca,Clavier,12,250
17,2025-02-02,Rabat,Telephone,2,3200
18,2025-02-03,Agadir,Laptop,1,9500
19,2025-02-04,Marrakech,Souris,,120
20,2025-02-05,Rabat,Tablette,3,2800
21,2025-02-07,Marrakech,Laptop,2,9500
22,2025-02-09,Fes,Clavier,5,250



In [44]:
brut = sc.textFile(os.path.join(DATA, "ventes.csv"), 3)

print("Nombre de lignes :", brut.count())
print("Nombre de partitions :", brut.getNumPartitions())
print("Première ligne :", brut.first())

Nombre de lignes : 23
Nombre de partitions : 3
Première ligne : id,date,ville,produit,quantite,prix


Nombre de lignes sans en-tête : 22
Premières lignes : ['1,2025-01-05,Casablanca,Laptop,2,9500', '2,2025-01-06,Rabat,Telephone,5,3200', '3,2025-01-07,Casablanca,Telephone,3,3200', '4,2025-01-09,Fes,Tablette,4,2800', '5,2025-01-11,Rabat,Laptop,1,9500']


In [ ]:
def parse(ligne):
    try:
        champs = ligne.split(',')
        id = int(champs[0])
        date = str(champs[1])
        ville = str(champs[2])
        produit = str(champs[3])
        quantite = int(champs[4])
        prix = float(champs[5])

        return [(id,date,ville,produit,quantite,prix)]

    except(ValueError,IndexError):
        return []

Pour Tester

In [26]:
print(parse("1,2025-01-05,Casablanca,Laptop,2,9500"))

print(parse("12,ligne_corrompue"))

print(parse("13,2025-01-22,Casablanca,Tablette,2,abc"))

[(1, '2025-01-05', 'Casablanca', 'Laptop', 2, 9500.0)]
[]
[]


In [34]:
ventes = donnees.flatMap(parse).cache()

print("Nombre de ventes valides :", ventes.count())

print("Premières ventes :")
for vente in ventes.take(5):
    print(vente)

print("Nombre de partitions :", ventes.getNumPartitions())

Nombre de ventes valides : 19
Premières ventes :
[(1, '2025-01-05', 'Casablanca', 'Laptop', 2, 9500.0), (2, '2025-01-06', 'Rabat', 'Telephone', 5, 3200.0), (3, '2025-01-07', 'Casablanca', 'Telephone', 3, 3200.0), (4, '2025-01-09', 'Fes', 'Tablette', 4, 2800.0), (5, '2025-01-11', 'Rabat', 'Laptop', 1, 9500.0)]
Nombre de partitions : 3


In [32]:
nombre_rejetees = donnees.count() - ventes.count()

print("Lignes rejetées :", nombre_rejetees)

Lignes rejetées : 3


flatMap() nettoie et transforme les données, tandis que cache() optimise leur réutilisation.

In [35]:
ca = ventes.map(lambda v: (v[2], v[4] * v[5]))

ca.take(5)

[('Casablanca', 19000.0),
 ('Rabat', 16000.0),
 ('Casablanca', 9600.0),
 ('Fes', 11200.0),
 ('Rabat', 9500.0)]

In [36]:
ca_ville = ca.reduceByKey(lambda a, b: a + b)

ca_ville.collect()

[('Casablanca', 34000.0),
 ('Fes', 35050.0),
 ('Rabat', 42100.0),
 ('Marrakech', 48600.0),
 ('Agadir', 14000.0)]

In [37]:
ca_ville_group = (
    ca.groupByKey()
      .mapValues(sum)
)

ca_ville_group.collect()

[('Casablanca', 34000.0),
 ('Fes', 35050.0),
 ('Rabat', 42100.0),
 ('Marrakech', 48600.0),
 ('Agadir', 14000.0)]

À retenir : pour calculer une somme, une moyenne ou un maximum par clé, privilégie généralement reduceByKey() ou aggregateByKey() plutôt que groupByKey().

In [38]:
print("=== reduceByKey ===")
print(ca_ville.toDebugString().decode())

print("=== groupByKey ===")
print(ca_ville_group.toDebugString().decode())

=== reduceByKey ===
(3) PythonRDD[44] at collect at /tmp/ipykernel_278/4154030656.py:3 []
 |  MapPartitionsRDD[43] at mapPartitions at PythonRDD.scala:168 []
 |  ShuffledRDD[42] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(3) PairwiseRDD[41] at reduceByKey at /tmp/ipykernel_278/4154030656.py:1 []
    |  PythonRDD[40] at reduceByKey at /tmp/ipykernel_278/4154030656.py:1 []
    |  PythonRDD[36] at RDD at PythonRDD.scala:56 []
    |      CachedPartitions: 3; MemorySize: 887.0 B; DiskSize: 0.0 B
    |  /home/jovyan/data/ventes.csv MapPartitionsRDD[14] at textFile at NativeMethodAccessorImpl.java:0 []
    |  /home/jovyan/data/ventes.csv HadoopRDD[13] at textFile at NativeMethodAccessorImpl.java:0 []
=== groupByKey ===
(3) PythonRDD[49] at collect at /tmp/ipykernel_278/2307940244.py:6 []
 |  MapPartitionsRDD[48] at mapPartitions at PythonRDD.scala:168 []
 |  ShuffledRDD[47] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(3) PairwiseRDD[46] at groupByKey at /tmp/ipykernel

In [39]:
nombre_ventes_ville = ventes.map(lambda v: (v[2], 1))

resultat = nombre_ventes_ville.countByKey()

print(resultat)

defaultdict(<class 'int'>, {'Casablanca': 4, 'Rabat': 5, 'Fes': 4, 'Marrakech': 3, 'Agadir': 3})


In [40]:
qte_produit = ventes.map(lambda v: (v[3], v[4]))

qte_produit.take(5)

[('Laptop', 2),
 ('Telephone', 5),
 ('Telephone', 3),
 ('Tablette', 4),
 ('Laptop', 1)]

In [41]:
max_qte = qte_produit.aggregateByKey(
    0,
    lambda acc, q: max(acc, q),
    lambda a, b: max(a, b)
)

max_qte.collect()

[('Tablette', 6),
 ('Clavier', 12),
 ('Laptop', 2),
 ('Telephone', 5),
 ('Souris', 30)]

In [18]:
ca_euro = ca_ville.mapValues(lambda montant: montant * 0.092)

ca_euro.collect()

NameError: name 'ca_ville' is not defined

In [20]:
ca_trie = ca_ville.sortBy(lambda x: x[1], ascending=False)

ca_trie.collect()

NameError: name 'ca_ville' is not defined

In [ ]:
ca_trie_2 = (
    ca_ville
    .map(lambda x: (x[1], x[0]))
    .sortByKey(ascending=False)
    .map(lambda x: (x[1], x[0]))
)

ca_trie_2.collect()

In [19]:
regions = sc.parallelize([
    ("Casablanca", "Casablanca-Settat"),
    ("Rabat", "Rabat-Sale-Kenitra"),
    ("Fes", "Fes-Meknes"),
    ("Tanger", "Tanger-Tetouan")
], 2)

regions.collect()

[('Casablanca', 'Casablanca-Settat'),
 ('Rabat', 'Rabat-Sale-Kenitra'),
 ('Fes', 'Fes-Meknes'),
 ('Tanger', 'Tanger-Tetouan')]

In [21]:
ventes_test = sc.parallelize([
    ("Casa", 1000),
    ("Rabat", 2000)
])

regions_test = sc.parallelize([
    ("Casa", "Casablanca-Settat"),
    ("Rabat", "Rabat-Sale-Kenitra")
])

ventes_test.join(regions_test).collect()

[('Rabat', (2000, 'Rabat-Sale-Kenitra')),
 ('Casa', (1000, 'Casablanca-Settat'))]